# 03b - Multi-Year Historical Data Preparation

## Objective

This notebook prepares the multi-year CAMS solar-radiation dataset for climatological analysis of solar availability anomalies and solar drought events.

The purpose of this stage is to establish whether the historical dataset is sufficiently complete, temporally consistent, and internally coherent to support construction of a multi-year solar-availability reference distribution.

The preparation workflow includes:

1. Loading the multi-year CAMS solar-radiation dataset.
2. Verifying the dataset metadata and requested study location.
3. Parsing and validating observation periods.
4. Establishing the exact temporal coverage.
5. Checking hourly temporal continuity and duplicate observations.
6. Assessing missing values and numerical data quality.
7. Evaluating the reliability information provided by CAMS.
8. Checking physical and component-level consistency of the radiation variables.
9. Characterizing nighttime, transition, and solar-active periods.
10. Applying the clear-sky normalization methodology established previously.
11. Constructing the multi-year Clear-Sky Index (CSI) series.
12. Separating the primary 2015–2024 analysis period from observations outside that period.

## Primary Analysis Period

The primary historical analysis period is defined as the ten complete calendar years:

$$
2015\text{-}01\text{-}01
\leq t <
2025\text{-}01\text{-}01
$$

Observations outside this interval are retained but are not included in the primary climatological analysis.

## Research Principle

This notebook performs historical data preparation and quality assessment.

No final anomaly threshold or solar-drought threshold is defined here.

Observations are not removed solely because they exhibit unusually low solar availability.

Potential quality issues are identified using explicit diagnostic criteria and retained as quality information wherever possible.

## Dataset Metadata

The historical dataset was obtained from the CAMS Radiation Service and contains hourly solar-radiation quantities for the Davos study location.

The metadata embedded in the downloaded file specifies:

- Latitude: 46.8000° N
- Longitude: 9.8300° E
- Altitude: 1610 m
- Time reference: Universal Time (UT)
- Temporal resolution: 1 hour

The dataset provides:

- TOA irradiation
- Clear-sky GHI
- Clear-sky BHI
- Clear-sky DHI
- Clear-sky BNI
- All-sky GHI
- All-sky BHI
- All-sky DHI
- All-sky BNI
- Reliability information

The primary solar-availability variable is Global Horizontal Irradiation (GHI).

The corresponding clear-sky GHI will be used as the normalization reference:

$$
CSI_t =
\frac{GHI_{\text{observed},t}}
{GHI_{\text{clear},t}}
$$

The CSI methodology was previously developed and validated in Notebook 03 using the January 2020 pilot dataset.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
data_path = Path(
    "../data/raw/solar/f221f71b2cfe6477f411ad39b80623d.csv"
)

print("File exists:", data_path.exists())
print("File:", data_path)

File exists: True
File: ..\data\raw\solar\f221f71b2cfe6477f411ad39b80623d.csv


In [4]:
df = pd.read_csv(
    data_path,
    sep=";",
    comment="#",
    header=None,
    names=[
        "observation_period",
        "toa",
        "clear_sky_ghi",
        "clear_sky_bhi",
        "clear_sky_dhi",
        "clear_sky_bni",
        "ghi",
        "bhi",
        "dhi",
        "bni",
        "reliability",
    ],
)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Shape: (87696, 11)

Columns:
['observation_period', 'toa', 'clear_sky_ghi', 'clear_sky_bhi', 'clear_sky_dhi', 'clear_sky_bni', 'ghi', 'bhi', 'dhi', 'bni', 'reliability']


In [5]:
periods = df["observation_period"].str.split("/", expand=True)

df["start_time"] = pd.to_datetime(periods[0])
df["end_time"] = pd.to_datetime(periods[1])

df = df.sort_values("start_time").reset_index(drop=True)

print("========== TIMESTAMP PARSING ==========")

print("\nFirst observation:")
print(df.iloc[0]["observation_period"])

print("\nLast observation:")
print(df.iloc[-1]["observation_period"])

print("\nFirst start time:")
print(df["start_time"].iloc[0])

print("\nLast start time:")
print(df["start_time"].iloc[-1])

print("\nFirst end time:")
print(df["end_time"].iloc[0])

print("\nLast end time:")
print(df["end_time"].iloc[-1])

========== TIMESTAMP PARSING ==========

First observation:
2015-01-01T00:00:00.0/2015-01-01T01:00:00.0

Last observation:
2025-01-01T23:00:00.0/2025-01-02T00:00:00.0

First start time:
2015-01-01 00:00:00

Last start time:
2025-01-01 23:00:00

First end time:
2015-01-01 01:00:00

Last end time:
2025-01-02 00:00:00


In [6]:
duration_hours = (
    (df["end_time"] - df["start_time"])
    .dt.total_seconds() / 3600
)

print("========== OBSERVATION DURATION ==========")

print(duration_hours.value_counts().sort_index())

print("\nUnexpected durations:")
print((duration_hours != 1.0).sum())

========== OBSERVATION DURATION ==========
1.0    87696
Name: count, dtype: int64

Unexpected durations:
0


In [7]:
print("=" * 70)
print("COMPREHENSIVE HISTORICAL DATASET QA")
print("=" * 70)

# ------------------------------------------------------------
# 1. DATASET STRUCTURE
# ------------------------------------------------------------

print("\n1. DATASET STRUCTURE")

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 2. TEMPORAL COVERAGE
# ------------------------------------------------------------

print("\n2. TEMPORAL COVERAGE")

print("First start time:", df["start_time"].min())
print("Last start time:", df["start_time"].max())

print("First end time:", df["end_time"].min())
print("Last end time:", df["end_time"].max())


# ------------------------------------------------------------
# 3. OBSERVATION DURATION
# ------------------------------------------------------------

print("\n3. OBSERVATION DURATION")

duration_hours = (
    (df["end_time"] - df["start_time"])
    .dt.total_seconds() / 3600
)

print(duration_hours.value_counts().sort_index())

print(
    "Unexpected durations:",
    (duration_hours != 1.0).sum()
)


# ------------------------------------------------------------
# 4. TEMPORAL CONTINUITY
# ------------------------------------------------------------

print("\n4. TEMPORAL CONTINUITY")

time_difference = df["start_time"].diff()

unexpected_intervals = (
    time_difference.iloc[1:] != pd.Timedelta(hours=1)
)

print(
    "Unexpected intervals:",
    unexpected_intervals.sum()
)

print(
    "Hourly intervals:",
    (time_difference.iloc[1:] == pd.Timedelta(hours=1)).sum()
)


# ------------------------------------------------------------
# 5. DUPLICATE OBSERVATIONS
# ------------------------------------------------------------

print("\n5. DUPLICATE OBSERVATIONS")

print(
    "Duplicate observation periods:",
    df["observation_period"].duplicated().sum()
)

print(
    "Duplicate start timestamps:",
    df["start_time"].duplicated().sum()
)

print(
    "Duplicate end timestamps:",
    df["end_time"].duplicated().sum()
)


# ------------------------------------------------------------
# 6. MISSING VALUES
# ------------------------------------------------------------

print("\n6. MISSING VALUES")

numeric_columns = [
    "toa",
    "clear_sky_ghi",
    "clear_sky_bhi",
    "clear_sky_dhi",
    "clear_sky_bni",
    "ghi",
    "bhi",
    "dhi",
    "bni",
    "reliability",
]

missing_values = df[numeric_columns].isna().sum()

print(missing_values)

print(
    "\nTotal missing numeric values:",
    missing_values.sum()
)


# ------------------------------------------------------------
# 7. NEGATIVE RADIATION VALUES
# ------------------------------------------------------------

print("\n7. NEGATIVE RADIATION VALUES")

radiation_columns = [
    "toa",
    "clear_sky_ghi",
    "clear_sky_bhi",
    "clear_sky_dhi",
    "clear_sky_bni",
    "ghi",
    "bhi",
    "dhi",
    "bni",
]

for column in radiation_columns:
    print(
        f"{column}:",
        (df[column] < 0).sum()
    )


# ------------------------------------------------------------
# 8. RELIABILITY
# ------------------------------------------------------------

print("\n8. RELIABILITY")

print(
    df["reliability"].describe()
)

print(
    "\nReliability < 1.0:",
    (df["reliability"] < 1.0).sum()
)

print(
    "Reliability < 0.9:",
    (df["reliability"] < 0.9).sum()
)

print(
    "Reliability < 0.8:",
    (df["reliability"] < 0.8).sum()
)

print(
    "Reliability <= 0.5:",
    (df["reliability"] <= 0.5).sum()
)


# ------------------------------------------------------------
# 9. GHI COMPONENT CONSISTENCY
# ------------------------------------------------------------

print("\n9. GHI COMPONENT CONSISTENCY")

df["ghi_component_difference"] = (
    df["ghi"] - (df["bhi"] + df["dhi"])
)

print(
    df["ghi_component_difference"].describe()
)

print(
    "\nAbsolute difference > 0.01:",
    (
        df["ghi_component_difference"].abs() > 0.01
    ).sum()
)

print(
    "Maximum absolute difference:",
    df["ghi_component_difference"].abs().max()
)


# ------------------------------------------------------------
# 10. OBSERVED VS CLEAR-SKY
# ------------------------------------------------------------

print("\n10. OBSERVED VS CLEAR-SKY")

clear_sky_pairs = [
    ("ghi", "clear_sky_ghi"),
    ("bhi", "clear_sky_bhi"),
    ("dhi", "clear_sky_dhi"),
    ("bni", "clear_sky_bni"),
]

for observed, clear_sky in clear_sky_pairs:

    count = (
        df[observed] > df[clear_sky]
    ).sum()

    print(
        f"{observed} > {clear_sky}:",
        count
    )


# ------------------------------------------------------------
# 11. NIGHTTIME BEHAVIOR
# ------------------------------------------------------------

print("\n11. NIGHTTIME BEHAVIOR")

night_mask = (
    df["clear_sky_ghi"] == 0
)

daylight_mask = (
    df["clear_sky_ghi"] > 0
)

print(
    "Nighttime observations:",
    night_mask.sum()
)

print(
    "Positive-reference observations:",
    daylight_mask.sum()
)

print(
    "Nighttime observations with positive GHI:",
    (
        df.loc[night_mask, "ghi"] > 0
    ).sum()
)

print(
    "Daylight observations with zero GHI:",
    (
        df.loc[daylight_mask, "ghi"] == 0
    ).sum()
)


# ------------------------------------------------------------
# 12. ZERO VALUES IN RADIATION COMPONENTS
# ------------------------------------------------------------

print("\n12. ZERO VALUES")

for column in radiation_columns:

    print(
        f"{column}:",
        (df[column] == 0).sum()
    )


# ------------------------------------------------------------
# 13. BASIC STATISTICS — GHI
# ------------------------------------------------------------

print("\n13. GHI DISTRIBUTION")

print(
    df["ghi"].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# ------------------------------------------------------------
# 14. BASIC STATISTICS — CLEAR-SKY GHI
# ------------------------------------------------------------

print("\n14. CLEAR-SKY GHI DISTRIBUTION")

print(
    df["clear_sky_ghi"].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# ------------------------------------------------------------
# 15. DATA TYPE CHECK
# ------------------------------------------------------------

print("\n15. DATA TYPES")

print(df.dtypes)


# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("QA SUMMARY")
print("=" * 70)

print(
    "Missing numeric values:",
    missing_values.sum()
)

print(
    "Unexpected durations:",
    (duration_hours != 1.0).sum()
)

print(
    "Unexpected temporal intervals:",
    unexpected_intervals.sum()
)

print(
    "Duplicate start timestamps:",
    df["start_time"].duplicated().sum()
)

print(
    "Negative radiation values:",
    sum(
        (df[column] < 0).sum()
        for column in radiation_columns
    )
)

print(
    "GHI component inconsistencies > 0.01:",
    (
        df["ghi_component_difference"].abs() > 0.01
    ).sum()
)

print(
    "Nighttime observations:",
    night_mask.sum()
)

print(
    "Positive clear-sky reference observations:",
    daylight_mask.sum()
)

print("=" * 70)

COMPREHENSIVE HISTORICAL DATASET QA

1. DATASET STRUCTURE
Rows: 87696
Columns: 13

Column names:
['observation_period', 'toa', 'clear_sky_ghi', 'clear_sky_bhi', 'clear_sky_dhi', 'clear_sky_bni', 'ghi', 'bhi', 'dhi', 'bni', 'reliability', 'start_time', 'end_time']

2. TEMPORAL COVERAGE
First start time: 2015-01-01 00:00:00
Last start time: 2025-01-01 23:00:00
First end time: 2015-01-01 01:00:00
Last end time: 2025-01-02 00:00:00

3. OBSERVATION DURATION
1.0    87696
Name: count, dtype: int64
Unexpected durations: 0

4. TEMPORAL CONTINUITY
Unexpected intervals: 0
Hourly intervals: 87695

5. DUPLICATE OBSERVATIONS
Duplicate observation periods: 0
Duplicate start timestamps: 0
Duplicate end timestamps: 0

6. MISSING VALUES
toa              0
clear_sky_ghi    0
clear_sky_bhi    0
clear_sky_dhi    0
clear_sky_bni    0
ghi              0
bhi              0
dhi              0
bni              0
reliability      0
dtype: int64

Total missing numeric values: 0

7. NEGATIVE RADIATION VALUES
toa: 

In [8]:
print("\n" + "=" * 70)
print("FINAL HISTORICAL QA SUMMARY")
print("=" * 70)

print("\nDataset:")
print("Total observations:", len(df))

print("\nTemporal:")
print("First observation:", df["start_time"].min())
print("Last observation:", df["end_time"].max())
print("Unexpected durations:", (duration_hours != 1.0).sum())
print("Unexpected hourly intervals:", unexpected_intervals.sum())
print("Duplicate start timestamps:", df["start_time"].duplicated().sum())

print("\nMissing values:")
print("Total missing numeric values:", df[numeric_columns].isna().sum().sum())

print("\nNegative radiation values:")
print(
    "Total:",
    sum(
        (df[column] < 0).sum()
        for column in radiation_columns
    )
)

print("\nRadiation consistency:")
print(
    "GHI component inconsistencies > 0.01:",
    (df["ghi_component_difference"].abs() > 0.01).sum()
)

print("\nClear-sky reference:")
print(
    "Zero clear-sky GHI:",
    (df["clear_sky_ghi"] == 0).sum()
)

print(
    "Positive clear-sky GHI:",
    (df["clear_sky_ghi"] > 0).sum()
)

print("\nReliability:")
print(
    "Reliability < 1.0:",
    (df["reliability"] < 1.0).sum()
)

print(
    "Reliability < 0.9:",
    (df["reliability"] < 0.9).sum()
)

print(
    "Reliability < 0.8:",
    (df["reliability"] < 0.8).sum()
)

print("\nNighttime:")
print(
    "Nighttime observations with positive GHI:",
    (
        df.loc[df["clear_sky_ghi"] == 0, "ghi"] > 0
    ).sum()
)

print("=" * 70)


FINAL HISTORICAL QA SUMMARY

Dataset:
Total observations: 87696

Temporal:
First observation: 2015-01-01 00:00:00
Last observation: 2025-01-02 00:00:00
Unexpected durations: 0
Unexpected hourly intervals: 0
Duplicate start timestamps: 0

Missing values:
Total missing numeric values: 0

Negative radiation values:
Total: 0

Radiation consistency:
GHI component inconsistencies > 0.01: 0

Clear-sky reference:
Zero clear-sky GHI: 40066
Positive clear-sky GHI: 47630

Reliability:
Reliability < 1.0: 11722
Reliability < 0.9: 7923
Reliability < 0.8: 4594

Nighttime:
Nighttime observations with positive GHI: 0


In [9]:
print("=" * 70)
print("RELIABILITY AND SOLAR-PERIOD QA")
print("=" * 70)

# ------------------------------------------------------------
# 1. CLEAR-SKY REFERENCE
# ------------------------------------------------------------

print("\n1. CLEAR-SKY GHI")

night_mask = df["clear_sky_ghi"] == 0
daylight_mask = df["clear_sky_ghi"] > 0

print("Zero clear-sky GHI:", night_mask.sum())
print("Positive clear-sky GHI:", daylight_mask.sum())

print(
    "Other clear-sky GHI values:",
    (~(night_mask | daylight_mask)).sum()
)


# ------------------------------------------------------------
# 2. RELIABILITY DISTRIBUTION
# ------------------------------------------------------------

print("\n2. RELIABILITY DISTRIBUTION")

print(df["reliability"].describe())

print(
    "\nReliability = 1.0:",
    (df["reliability"] == 1.0).sum()
)

print(
    "Reliability < 1.0:",
    (df["reliability"] < 1.0).sum()
)

print(
    "Reliability < 0.9:",
    (df["reliability"] < 0.9).sum()
)

print(
    "Reliability < 0.8:",
    (df["reliability"] < 0.8).sum()
)

print(
    "Reliability <= 0.5:",
    (df["reliability"] <= 0.5).sum()
)


# ------------------------------------------------------------
# 3. RELIABILITY BY SOLAR PERIOD
# ------------------------------------------------------------

print("\n3. RELIABILITY BY SOLAR PERIOD")

df["solar_period"] = np.where(
    df["clear_sky_ghi"] == 0,
    "nighttime",
    "positive_reference"
)

reliability_by_period = (
    df.groupby("solar_period")["reliability"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        maximum="max"
    )
)

print(reliability_by_period)


# ------------------------------------------------------------
# 4. REDUCED RELIABILITY BY SOLAR PERIOD
# ------------------------------------------------------------

print("\n4. REDUCED RELIABILITY BY SOLAR PERIOD")

reduced_reliability = df["reliability"] < 1.0

print(
    pd.crosstab(
        df["solar_period"],
        reduced_reliability
    )
)


# ------------------------------------------------------------
# 5. GHI BY RELIABILITY — POSITIVE REFERENCE ONLY
# ------------------------------------------------------------

print("\n5. GHI BY RELIABILITY — POSITIVE REFERENCE ONLY")

daylight_df = df[daylight_mask].copy()

daylight_df["reliability_flag"] = np.where(
    daylight_df["reliability"] < 1.0,
    "reduced_reliability",
    "full_reliability"
)

print(
    daylight_df.groupby("reliability_flag")["ghi"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        maximum="max"
    )
)


# ------------------------------------------------------------
# 6. CLEAR-SKY GHI BY RELIABILITY — POSITIVE REFERENCE ONLY
# ------------------------------------------------------------

print(
    "\n6. CLEAR-SKY GHI BY RELIABILITY — POSITIVE REFERENCE ONLY"
)

print(
    daylight_df.groupby("reliability_flag")["clear_sky_ghi"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        maximum="max"
    )
)


# ------------------------------------------------------------
# 7. NIGHTTIME PHYSICAL CHECK
# ------------------------------------------------------------

print("\n7. NIGHTTIME PHYSICAL CHECK")

print(
    "Nighttime rows:",
    night_mask.sum()
)

print(
    "Nighttime rows with GHI > 0:",
    (df.loc[night_mask, "ghi"] > 0).sum()
)

print(
    "Nighttime rows with BHI > 0:",
    (df.loc[night_mask, "bhi"] > 0).sum()
)

print(
    "Nighttime rows with DHI > 0:",
    (df.loc[night_mask, "dhi"] > 0).sum()
)

print(
    "Nighttime rows with BNI > 0:",
    (df.loc[night_mask, "bni"] > 0).sum()
)

print("=" * 70)

RELIABILITY AND SOLAR-PERIOD QA

1. CLEAR-SKY GHI
Zero clear-sky GHI: 40066
Positive clear-sky GHI: 47630
Other clear-sky GHI values: 0

2. RELIABILITY DISTRIBUTION
count    87696.000000
mean         0.976896
std          0.070011
min          0.500000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000
Name: reliability, dtype: float64

Reliability = 1.0: 75974
Reliability < 1.0: 11722
Reliability < 0.9: 7923
Reliability < 0.8: 4594
Reliability <= 0.5: 16

3. RELIABILITY BY SOLAR PERIOD
                    count      mean  median  minimum  q25  q75  maximum
solar_period                                                           
nighttime           40066  0.999988     1.0   0.9917  1.0  1.0      1.0
positive_reference  47630  0.957470     1.0   0.5000  1.0  1.0      1.0

4. REDUCED RELIABILITY BY SOLAR PERIOD
reliability         False  True 
solar_period                    
nighttime           40010     56
positive_reference  35964  11666

5. GHI BY 

In [10]:
print("=" * 70)
print("RELIABILITY CONCENTRATION ANALYSIS")
print("=" * 70)

# Reduced reliability
reduced = df["reliability"] < 1.0

# Solar-period counts
reliability_counts = pd.crosstab(
    df["solar_period"],
    reduced
)

print("\n1. RELIABILITY BY SOLAR PERIOD")
print(reliability_counts)


# ------------------------------------------------------------
# Percent of reduced-reliability observations by period
# ------------------------------------------------------------

reduced_by_period = (
    df.loc[reduced, "solar_period"]
    .value_counts()
)

print("\n2. REDUCED-RELIABILITY OBSERVATIONS BY PERIOD")
print(reduced_by_period)

print("\nPercentage of all reduced-reliability observations:")
print(
    100 *
    reduced_by_period /
    reduced.sum()
)


# ------------------------------------------------------------
# Reliability < 0.9
# ------------------------------------------------------------

low_reliability_09 = df["reliability"] < 0.9

print("\n3. RELIABILITY < 0.9 BY SOLAR PERIOD")

print(
    pd.crosstab(
        df["solar_period"],
        low_reliability_09
    )
)


# ------------------------------------------------------------
# Reliability < 0.8
# ------------------------------------------------------------

low_reliability_08 = df["reliability"] < 0.8

print("\n4. RELIABILITY < 0.8 BY SOLAR PERIOD")

print(
    pd.crosstab(
        df["solar_period"],
        low_reliability_08
    )
)


# ------------------------------------------------------------
# Reliability distribution — positive-reference observations
# ------------------------------------------------------------

print("\n5. RELIABILITY — POSITIVE REFERENCE ONLY")

print(
    df.loc[
        daylight_mask,
        "reliability"
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# ------------------------------------------------------------
# Reduced reliability percentage within each period
# ------------------------------------------------------------

print("\n6. REDUCED-RELIABILITY RATE WITHIN EACH PERIOD")

period_reliability_rate = (
    df.groupby("solar_period")["reliability"]
    .apply(lambda x: (x < 1.0).mean() * 100)
)

print(period_reliability_rate)


print("=" * 70)

RELIABILITY CONCENTRATION ANALYSIS

1. RELIABILITY BY SOLAR PERIOD
reliability         False  True 
solar_period                    
nighttime           40010     56
positive_reference  35964  11666

2. REDUCED-RELIABILITY OBSERVATIONS BY PERIOD
solar_period
positive_reference    11666
nighttime                56
Name: count, dtype: int64

Percentage of all reduced-reliability observations:
solar_period
positive_reference    99.522266
nighttime              0.477734
Name: count, dtype: float64

3. RELIABILITY < 0.9 BY SOLAR PERIOD
reliability         False  True 
solar_period                    
nighttime           40066      0
positive_reference  39707   7923

4. RELIABILITY < 0.8 BY SOLAR PERIOD
reliability         False  True 
solar_period                    
nighttime           40066      0
positive_reference  43036   4594

5. RELIABILITY — POSITIVE REFERENCE ONLY
count    47630.000000
mean         0.957470
std          0.090547
min          0.500000
1%           0.633300
5%       

## Reliability Assessment and Retention Policy

The historical dataset contains a reliability value for every hourly observation.

Across the complete downloaded dataset:

- 75,974 observations have reliability equal to 1.0.
- 11,722 observations have reliability below 1.0.
- 7,923 observations have reliability below 0.9.
- 4,594 observations have reliability below 0.8.
- 16 observations have reliability less than or equal to 0.5.

Reduced reliability is strongly concentrated in observations with a positive clear-sky GHI reference.

Of the 11,722 observations with reliability below 1.0, 11,666 (99.52%) occur during positive-reference periods, while 56 (0.48%) occur during nighttime periods.

Within the positive-reference population, 24.49% of observations have reliability below 1.0, compared with 0.14% of nighttime observations.

This concentration indicates that the reliability variable is not independent of the solar-observation period. Therefore, observations with reduced reliability are not automatically removed from the analysis.

Instead, reliability is retained as a quality-control variable and represented through an explicit diagnostic flag.

No observation is excluded solely because its reliability value is below 1.0.

This retention policy is important because unusually low solar availability is itself a scientific quantity of interest in the solar-drought analysis. Automatically removing reduced-reliability observations could preferentially remove observations from the solar-active population and potentially alter the lower tail of the solar-availability distribution.

The effect of reliability on subsequent anomaly and drought results will be evaluated through sensitivity analysis rather than assumed a priori.

In [11]:
print("=" * 70)
print("RADIATION COMPONENT CONSISTENCY ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. GHI = BHI + DHI
# ------------------------------------------------------------

df["ghi_component_difference"] = (
    df["ghi"] - (df["bhi"] + df["dhi"])
)

print("\n1. GHI COMPONENT BALANCE")

print(
    df["ghi_component_difference"].describe()
)

print(
    "\nAbsolute difference > 0.01:",
    (
        df["ghi_component_difference"].abs() > 0.01
    ).sum()
)

print(
    "Maximum absolute difference:",
    df["ghi_component_difference"].abs().max()
)


# ------------------------------------------------------------
# 2. OBSERVED > CLEAR-SKY
# ------------------------------------------------------------

print("\n2. OBSERVED RADIATION ABOVE CLEAR-SKY")

df["ghi_above_clear"] = (
    df["ghi"] > df["clear_sky_ghi"]
)

df["bhi_above_clear"] = (
    df["bhi"] > df["clear_sky_bhi"]
)

df["dhi_above_clear"] = (
    df["dhi"] > df["clear_sky_dhi"]
)

df["bni_above_clear"] = (
    df["bni"] > df["clear_sky_bni"]
)

for column in [
    "ghi_above_clear",
    "bhi_above_clear",
    "dhi_above_clear",
    "bni_above_clear"
]:
    print(
        f"{column}:",
        df[column].sum()
    )


# ------------------------------------------------------------
# 3. ANY COMPONENT ABOVE CLEAR-SKY
# ------------------------------------------------------------

any_component_above_clear = (
    df[
        [
            "ghi_above_clear",
            "bhi_above_clear",
            "dhi_above_clear",
            "bni_above_clear"
        ]
    ]
    .any(axis=1)
)

print(
    "\nAny component above clear-sky:",
    any_component_above_clear.sum()
)


# ------------------------------------------------------------
# 4. WHICH COMPONENTS EXCEED CLEAR-SKY?
# ------------------------------------------------------------

print("\n4. COMPONENT EXCEEDANCE PATTERNS")

exceedance_pattern = pd.DataFrame({
    "GHI": df["ghi_above_clear"],
    "BHI": df["bhi_above_clear"],
    "DHI": df["dhi_above_clear"],
    "BNI": df["bni_above_clear"]
})

print(
    exceedance_pattern.value_counts()
)


# ------------------------------------------------------------
# 5. EXCEEDANCES DURING POSITIVE-REFERENCE PERIODS
# ------------------------------------------------------------

print(
    "\n5. EXCEEDANCES — POSITIVE REFERENCE ONLY"
)

positive_reference = (
    df["clear_sky_ghi"] > 0
)

for column in [
    "ghi_above_clear",
    "bhi_above_clear",
    "dhi_above_clear",
    "bni_above_clear"
]:

    print(
        f"{column}:",
        df.loc[
            positive_reference,
            column
        ].sum()
    )


# ------------------------------------------------------------
# 6. NIGHTTIME COMPONENT CHECK
# ------------------------------------------------------------

print("\n6. NIGHTTIME COMPONENT CHECK")

nighttime = (
    df["clear_sky_ghi"] == 0
)

for column in [
    "ghi",
    "bhi",
    "dhi",
    "bni"
]:

    print(
        f"Nighttime {column} > 0:",
        (
            df.loc[
                nighttime,
                column
            ] > 0
        ).sum()
    )


# ------------------------------------------------------------
# 7. COMPONENT ZERO PATTERNS
# ------------------------------------------------------------

print("\n7. COMPONENT ZERO PATTERNS")

print(
    "GHI > 0 and BHI = 0:",
    (
        (df["ghi"] > 0) &
        (df["bhi"] == 0)
    ).sum()
)

print(
    "GHI > 0 and DHI = 0:",
    (
        (df["ghi"] > 0) &
        (df["dhi"] == 0)
    ).sum()
)

print(
    "GHI = 0 and BHI > 0:",
    (
        (df["ghi"] == 0) &
        (df["bhi"] > 0)
    ).sum()
)

print(
    "GHI = 0 and DHI > 0:",
    (
        (df["ghi"] == 0) &
        (df["dhi"] > 0)
    ).sum()
)

print(
    "BNI > 0 and BHI = 0:",
    (
        (df["bni"] > 0) &
        (df["bhi"] == 0)
    ).sum()
)

print("=" * 70)

RADIATION COMPONENT CONSISTENCY ANALYSIS

1. GHI COMPONENT BALANCE
count    8.769600e+04
mean     2.280606e-08
std      3.651379e-05
min     -1.000000e-04
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e-04
Name: ghi_component_difference, dtype: float64

Absolute difference > 0.01: 0
Maximum absolute difference: 0.00010000000008858478

2. OBSERVED RADIATION ABOVE CLEAR-SKY
ghi_above_clear: 1
bhi_above_clear: 0
dhi_above_clear: 27599
bni_above_clear: 0

Any component above clear-sky: 27599

4. COMPONENT EXCEEDANCE PATTERNS
GHI    BHI    DHI    BNI  
False  False  False  False    60097
              True   False    27598
True   False  True   False        1
Name: count, dtype: int64

5. EXCEEDANCES — POSITIVE REFERENCE ONLY
ghi_above_clear: 1
bhi_above_clear: 0
dhi_above_clear: 27599
bni_above_clear: 0

6. NIGHTTIME COMPONENT CHECK
Nighttime ghi > 0: 0
Nighttime bhi > 0: 0
Nighttime dhi > 0: 0
Nighttime bni > 0: 0

7. COMPONENT ZERO PATTERNS
GHI > 0 and

In [12]:
print("=" * 70)
print("DHI ABOVE CLEAR-SKY INVESTIGATION")
print("=" * 70)

dhi_excess = (
    df["dhi"] - df["clear_sky_dhi"]
)

dhi_excess_mask = (
    df["dhi"] > df["clear_sky_dhi"]
)

dhi_excess_df = df[dhi_excess_mask].copy()

# ------------------------------------------------------------
# 1. Number and percentage
# ------------------------------------------------------------

print("\n1. DHI EXCEEDANCE")

print(
    "Number of observations:",
    len(dhi_excess_df)
)

print(
    "Percentage of all observations:",
    100 * len(dhi_excess_df) / len(df)
)

print(
    "Percentage of positive-reference observations:",
    100 *
    len(
        dhi_excess_df[
            dhi_excess_df["clear_sky_ghi"] > 0
        ]
    )
    /
    (df["clear_sky_ghi"] > 0).sum()
)


# ------------------------------------------------------------
# 2. DHI EXCESS MAGNITUDE
# ------------------------------------------------------------

print("\n2. DHI EXCESS MAGNITUDE")

print(
    dhi_excess_df["dhi"]
    .sub(dhi_excess_df["clear_sky_dhi"])
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# ------------------------------------------------------------
# 3. DHI RATIO TO CLEAR-SKY DHI
# ------------------------------------------------------------

positive_clear_dhi = (
    dhi_excess_df["clear_sky_dhi"] > 0
)

dhi_excess_df["dhi_clear_ratio"] = np.nan

dhi_excess_df.loc[
    positive_clear_dhi,
    "dhi_clear_ratio"
] = (
    dhi_excess_df.loc[
        positive_clear_dhi,
        "dhi"
    ]
    /
    dhi_excess_df.loc[
        positive_clear_dhi,
        "clear_sky_dhi"
    ]
)

print("\n3. DHI / CLEAR-SKY DHI RATIO")

print(
    dhi_excess_df["dhi_clear_ratio"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# ------------------------------------------------------------
# 4. SOLAR PERIOD
# ------------------------------------------------------------

print("\n4. DHI EXCEEDANCES BY SOLAR PERIOD")

print(
    dhi_excess_df["solar_period"]
    .value_counts()
)


# ------------------------------------------------------------
# 5. GHI RELATIONSHIP
# ------------------------------------------------------------

print("\n5. GHI RELATIONSHIP")

print(
    "DHI exceedance + GHI above clear:",
    (
        dhi_excess_df["ghi"]
        >
        dhi_excess_df["clear_sky_ghi"]
    ).sum()
)

print(
    "DHI exceedance + GHI below/equal clear:",
    (
        dhi_excess_df["ghi"]
        <=
        dhi_excess_df["clear_sky_ghi"]
    ).sum()
)


# ------------------------------------------------------------
# 6. BHI RELATIONSHIP
# ------------------------------------------------------------

print("\n6. BHI RELATIONSHIP")

print(
    "DHI exceedance + BHI above clear:",
    (
        dhi_excess_df["bhi"]
        >
        dhi_excess_df["clear_sky_bhi"]
    ).sum()
)

print(
    "DHI exceedance + BHI below/equal clear:",
    (
        dhi_excess_df["bhi"]
        <=
        dhi_excess_df["clear_sky_bhi"]
    ).sum()
)


# ------------------------------------------------------------
# 7. RELIABILITY
# ------------------------------------------------------------

print("\n7. RELIABILITY OF DHI EXCEEDANCES")

print(
    dhi_excess_df["reliability"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

print(
    "\nReduced reliability:",
    (
        dhi_excess_df["reliability"] < 1.0
    ).sum()
)

print(
    "Full reliability:",
    (
        dhi_excess_df["reliability"] == 1.0
    ).sum()
)


# ------------------------------------------------------------
# 8. MONTHLY DISTRIBUTION
# ------------------------------------------------------------

print("\n8. DHI EXCEEDANCES BY MONTH")

dhi_excess_df["month"] = (
    dhi_excess_df["start_time"].dt.month
)

print(
    dhi_excess_df["month"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 9. EXAMPLES
# ------------------------------------------------------------

print("\n9. EXAMPLE OBSERVATIONS")

example_columns = [
    "start_time",
    "ghi",
    "clear_sky_ghi",
    "bhi",
    "clear_sky_bhi",
    "dhi",
    "clear_sky_dhi",
    "bni",
    "clear_sky_bni",
    "reliability"
]

print(
    dhi_excess_df[
        example_columns
    ]
    .head(10)
    .to_string(index=False)
)

print("=" * 70)

DHI ABOVE CLEAR-SKY INVESTIGATION

1. DHI EXCEEDANCE
Number of observations: 27599
Percentage of all observations: 31.471218755701514
Percentage of positive-reference observations: 57.9445727482679

2. DHI EXCESS MAGNITUDE
count    27599.000000
mean        95.395537
std         90.935256
min          0.000300
1%           0.163398
5%           1.342980
10%          3.991720
25%         19.237000
50%         66.319100
75%        151.269000
90%        238.337120
95%        282.062000
99%        342.960136
max        440.751400
dtype: float64

3. DHI / CLEAR-SKY DHI RATIO
count    27599.000000
mean         2.108285
std          0.982998
min          1.000015
1%           1.004298
5%           1.029790
10%          1.080425
25%          1.304719
50%          1.852092
75%          2.659265
90%          3.548218
95%          4.081302
99%          5.027064
max          7.163853
Name: dhi_clear_ratio, dtype: float64

4. DHI EXCEEDANCES BY SOLAR PERIOD
solar_period
positive_reference    27599
N

## Component-Level Clear-Sky Comparison

The observed radiation components were compared with their corresponding clear-sky reference values.

The GHI component balance was highly consistent across the complete dataset:

$$
GHI_t \approx BHI_t + DHI_t
$$

The maximum absolute difference was approximately 0.0001 Wh/m², and no observations exceeded the predefined 0.01 Wh/m² diagnostic tolerance.

Comparison with the clear-sky components showed:

- GHI exceeded clear-sky GHI in 1 observation.
- BHI exceeded clear-sky BHI in 0 observations.
- DHI exceeded clear-sky DHI in 27,599 observations.
- BNI exceeded clear-sky BNI in 0 observations.

DHI exceeded its corresponding clear-sky value in approximately 31.47% of all observations and 57.94% of observations with a positive clear-sky GHI reference.

Importantly, DHI exceedance does not generally correspond to total GHI exceeding its clear-sky reference. The GHI component balance remains internally consistent, while the observed GHI is generally below the clear-sky GHI.

Therefore, component-level exceedance of clear-sky DHI is treated as a diagnostic characteristic rather than an automatic data-quality failure.

Observations are not removed on the basis of DHI exceeding its clear-sky counterpart.

Because the primary solar-availability metric is based on GHI relative to clear-sky GHI, the GHI relationship is given greater importance for subsequent normalization.

In [13]:
print("=" * 70)
print("GHI ABOVE CLEAR-SKY — INDIVIDUAL INSPECTION")
print("=" * 70)

ghi_exceedance = df[
    df["ghi"] > df["clear_sky_ghi"]
].copy()

print("\nNumber of observations:")
print(len(ghi_exceedance))

columns_to_show = [
    "observation_period",
    "toa",
    "clear_sky_ghi",
    "clear_sky_bhi",
    "clear_sky_dhi",
    "clear_sky_bni",
    "ghi",
    "bhi",
    "dhi",
    "bni",
    "reliability"
]

print("\nObservation:")
print(
    ghi_exceedance[
        columns_to_show
    ].to_string(index=False)
)

print("=" * 70)

GHI ABOVE CLEAR-SKY — INDIVIDUAL INSPECTION

Number of observations:
1

Observation:
                         observation_period   toa  clear_sky_ghi  clear_sky_bhi  clear_sky_dhi  clear_sky_bni    ghi    bhi    dhi  bni  reliability
2022-01-30T06:00:00.0/2022-01-30T07:00:00.0 1.456         0.4877         0.1695         0.3182            0.0 0.4881 0.1597 0.3284  0.0          1.0


## GHI Clear-Sky Exceedance Assessment

Only one observation in the complete historical dataset has observed GHI greater than the corresponding clear-sky GHI.

The observation occurs during:

- 2022-01-30 06:00–07:00
- observed GHI = 0.4881 Wh/m²
- clear-sky GHI = 0.4877 Wh/m²
- reliability = 1.0

The absolute difference is only 0.0004 Wh/m², corresponding to approximately 0.08% relative to the clear-sky GHI.

The observed GHI is internally consistent with its measured beam and diffuse components:

$$
0.1597 + 0.3284 = 0.4881
$$

The corresponding clear-sky components sum to:

$$
0.1695 + 0.3182 = 0.4877
$$

Therefore, the observation is not considered a data-quality failure and is retained in the dataset.

No observation is removed solely because observed GHI marginally exceeds its clear-sky reference.

The clear-sky reference is therefore treated as a normalization reference rather than an absolute physical upper-bound filter for automatic data deletion.

In [14]:
print("=" * 70)
print("BNI > 0 WHILE BHI = 0 — INDIVIDUAL INSPECTION")
print("=" * 70)

bni_without_bhi = df[
    (df["bni"] > 0) &
    (df["bhi"] == 0)
].copy()

print("\nNumber of observations:")
print(len(bni_without_bhi))

print("\nBNI statistics:")
print(
    bni_without_bhi["bni"].describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nBNI values:")
print(
    bni_without_bhi["bni"]
    .sort_values()
    .head(20)
    .to_string(index=False)
)

print("\nMaximum BNI:")
print(bni_without_bhi["bni"].max())

print("\nReliability:")
print(
    bni_without_bhi["reliability"].describe()
)

print("\nSolar period:")
print(
    bni_without_bhi["solar_period"].value_counts()
)

print("\nExample observations:")
example_columns = [
    "observation_period",
    "clear_sky_ghi",
    "clear_sky_bhi",
    "clear_sky_bni",
    "ghi",
    "bhi",
    "dhi",
    "bni",
    "reliability"
]

print(
    bni_without_bhi[
        example_columns
    ].head(20).to_string(index=False)
)

print("=" * 70)

BNI > 0 WHILE BHI = 0 — INDIVIDUAL INSPECTION

Number of observations:
272

BNI statistics:
count    272.000000
mean       0.000203
std        0.000222
min        0.000100
1%         0.000100
5%         0.000100
10%        0.000100
25%        0.000100
50%        0.000100
75%        0.000200
90%        0.000500
95%        0.000700
99%        0.001058
max        0.001300
Name: bni, dtype: float64

BNI values:
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001
0.0001

Maximum BNI:
0.0013

Reliability:
count    272.00000
mean       0.90855
std        0.11773
min        0.55830
25%        0.84170
50%        1.00000
75%        1.00000
max        1.00000
Name: reliability, dtype: float64

Solar period:
solar_period
positive_reference    272
Name: count, dtype: int64

Example observations:
                         observation_period  clear_sky_ghi  clear_sky_bhi  clear_sky_bni      ghi  bhi      dhi    bni  relia

## BNI/BHI Zero-Pattern Assessment

A total of 272 observations contain positive BNI while BHI is equal to zero.

The BNI values in these observations are very small:

- median = 0.0001 Wh/m²
- 75th percentile = 0.0002 Wh/m²
- 90th percentile = 0.0005 Wh/m²
- 99th percentile = 0.001058 Wh/m²
- maximum = 0.0013 Wh/m²

The inspected observations therefore do not represent a substantial beam-normal radiation signal.

These observations are retained without modification or exclusion. No additional BNI threshold is introduced because the primary solar-availability analysis is based on GHI relative to clear-sky GHI.

The BNI/BHI zero-pattern is retained as a documented diagnostic characteristic of the source dataset rather than treated as an automatic quality failure.

## Solar-Period Classification

For the historical analysis, observations are classified according to the availability of the clear-sky GHI reference.

Two primary categories are used:

1. **Nighttime:** clear-sky GHI = 0
2. **Positive-reference period:** clear-sky GHI > 0

The positive-reference category includes both normal daylight observations and low-sun transition periods. These observations are retained rather than removed using an arbitrary irradiance threshold.

This classification is used to determine when the clear-sky-index calculation is mathematically defined.

In [15]:
# Final solar-period classification

df["solar_period"] = np.where(
    df["clear_sky_ghi"] == 0,
    "nighttime",
    "positive_reference"
)

print("=" * 70)
print("FINAL SOLAR-PERIOD CLASSIFICATION")
print("=" * 70)

print("\nCounts:")
print(df["solar_period"].value_counts())

print("\nPercentages:")
print(
    (df["solar_period"].value_counts(normalize=True) * 100)
    .round(4)
)

print("\nClear-sky GHI by solar period:")
print(
    df.groupby("solar_period")["clear_sky_ghi"]
    .describe()
)

print("=" * 70)

FINAL SOLAR-PERIOD CLASSIFICATION

Counts:
solar_period
positive_reference    47630
nighttime             40066
Name: count, dtype: int64

Percentages:
solar_period
positive_reference    54.3126
nighttime             45.6874
Name: proportion, dtype: float64

Clear-sky GHI by solar period:
                      count        mean         std    min         25%  \
solar_period                                                             
nighttime           40066.0    0.000000    0.000000  0.000    0.000000   
positive_reference  47630.0  422.740074  298.927634  0.007  157.662975   

                          50%       75%        max  
solar_period                                        
nighttime             0.00000    0.0000     0.0000  
positive_reference  395.27615  675.4465  1021.4567  


## Historical Clear-Sky Index Construction

The clear-sky index (CSI) is defined as:

$$
CSI_t =
\frac{GHI_t}{GHI_{\mathrm{clear},t}}
$$

The calculation is performed only when the clear-sky GHI reference is strictly positive.

For observations where the clear-sky GHI is zero, CSI is set to NaN because the ratio is mathematically undefined.

No arbitrary minimum clear-sky GHI threshold is imposed at this stage. Very small positive clear-sky references are retained and flagged for diagnostic analysis.

The resulting CSI is the primary normalized solar-availability variable for subsequent anomaly and drought analysis.

In [16]:
# Construct CSI

df["csi"] = np.nan

positive_reference = df["clear_sky_ghi"] > 0

df.loc[positive_reference, "csi"] = (
    df.loc[positive_reference, "ghi"]
    / df.loc[positive_reference, "clear_sky_ghi"]
)

print("=" * 70)
print("HISTORICAL CSI")
print("=" * 70)

print("\nTotal observations:", len(df))

print("Nighttime / undefined CSI:",
      df["csi"].isna().sum())

print("Positive-reference / defined CSI:",
      df["csi"].notna().sum())

print("\nCSI statistics:")
print(
    df.loc[df["csi"].notna(), "csi"].describe(
        percentiles=[
            0.01, 0.05, 0.10, 0.20,
            0.25, 0.50, 0.75,
            0.90, 0.95, 0.99
        ]
    )
)

print("\nCSI outside [0, 1]:")
print(
    (
        (df["csi"] < 0) |
        (df["csi"] > 1)
    ).sum()
)

print("=" * 70)

HISTORICAL CSI

Total observations: 87696
Nighttime / undefined CSI: 40066
Positive-reference / defined CSI: 47630

CSI statistics:
count    47630.000000
mean         0.681834
std          0.315732
min          0.061697
1%           0.088213
5%           0.132581
10%          0.179170
20%          0.323037
25%          0.393491
50%          0.773199
75%          1.000000
90%          1.000000
95%          1.000000
99%          1.000000
max          1.000820
Name: csi, dtype: float64

CSI outside [0, 1]:
1


In [17]:
# CSI reconstruction check

df["ghi_reconstructed"] = (
    df["csi"] * df["clear_sky_ghi"]
)

reconstruction_error = (
    df.loc[positive_reference, "ghi_reconstructed"]
    - df.loc[positive_reference, "ghi"]
)

print("=" * 70)
print("CSI RECONSTRUCTION CHECK")
print("=" * 70)

print("\nMaximum absolute reconstruction error:")
print(reconstruction_error.abs().max())

print("\nMean absolute reconstruction error:")
print(reconstruction_error.abs().mean())

print("\nNumber of errors > 1e-10:")
print(
    (reconstruction_error.abs() > 1e-10).sum()
)

print("=" * 70)

CSI RECONSTRUCTION CHECK

Maximum absolute reconstruction error:
5.684341886080802e-14

Mean absolute reconstruction error:
1.3574430134149945e-15

Number of errors > 1e-10:
0


In [18]:
# Inspect small positive clear-sky references

positive_ref_df = df[
    df["clear_sky_ghi"] > 0
].copy()

print("=" * 70)
print("SMALL POSITIVE CLEAR-SKY REFERENCES")
print("=" * 70)

print("\nClear-sky GHI statistics:")
print(
    positive_ref_df["clear_sky_ghi"].describe(
        percentiles=[
            0.001, 0.005, 0.01,
            0.025, 0.05, 0.10,
            0.25, 0.50
        ]
    )
)

for threshold in [0.1, 0.5, 1, 2, 5, 10, 20]:
    count = (
        positive_ref_df["clear_sky_ghi"] <= threshold
    ).sum()

    print(
        f"\nclear_sky_ghi <= {threshold}: "
        f"{count} observations "
        f"({100 * count / len(positive_ref_df):.4f}%)"
    )

print("=" * 70)

SMALL POSITIVE CLEAR-SKY REFERENCES

Clear-sky GHI statistics:
count    47630.000000
mean       422.740074
std        298.927634
min          0.007000
0.1%         0.013400
0.5%         0.061315
1%           0.183858
2.5%         0.926772
5%           4.386325
10%         22.305330
25%        157.662975
50%        395.276150
max       1021.456700
Name: clear_sky_ghi, dtype: float64

clear_sky_ghi <= 0.1: 328 observations (0.6886%)

clear_sky_ghi <= 0.5: 838 observations (1.7594%)

clear_sky_ghi <= 1: 1277 observations (2.6811%)

clear_sky_ghi <= 2: 1773 observations (3.7224%)

clear_sky_ghi <= 5: 2511 observations (5.2719%)

clear_sky_ghi <= 10: 3556 observations (7.4659%)

clear_sky_ghi <= 20: 4581 observations (9.6179%)


In [19]:
print("=" * 70)
print("CSI RANGE DIAGNOSTICS")
print("=" * 70)

csi_valid = df["csi"].dropna()

print("\nCSI < 0:")
print((csi_valid < 0).sum())

print("\nCSI == 0:")
print((csi_valid == 0).sum())

print("\n0 < CSI <= 1:")
print(
    ((csi_valid > 0) & (csi_valid <= 1)).sum()
)

print("\nCSI > 1:")
print((csi_valid > 1).sum())

print("\nCSI > 1.01:")
print((csi_valid > 1.01).sum())

print("\nCSI > 1.05:")
print((csi_valid > 1.05).sum())

print("\nCSI < 0.05:")
print((csi_valid < 0.05).sum())

print("\nCSI < 0.10:")
print((csi_valid < 0.10).sum())

print("\nCSI < 0.20:")
print((csi_valid < 0.20).sum())

print("=" * 70)

CSI RANGE DIAGNOSTICS

CSI < 0:
0

CSI == 0:
0

0 < CSI <= 1:
47629

CSI > 1:
1

CSI > 1.01:
0

CSI > 1.05:
0

CSI < 0.05:
0

CSI < 0.10:
835

CSI < 0.20:
5408


In [20]:
# Flag observations with small positive clear-sky reference

df["low_reference_flag"] = (
    (df["clear_sky_ghi"] > 0) &
    (df["clear_sky_ghi"] <= 20)
)

print("=" * 70)
print("LOW CLEAR-SKY REFERENCE FLAG")
print("=" * 70)

print("\nFlagged observations:")
print(df["low_reference_flag"].sum())

print("\nPercentage of all observations:")
print(
    100 * df["low_reference_flag"].mean()
)

print("\nPercentage of positive-reference observations:")
print(
    100 * df.loc[
        df["clear_sky_ghi"] > 0,
        "low_reference_flag"
    ].mean()
)

print("\nCSI statistics by reference category:")

print(
    df.loc[
        df["clear_sky_ghi"] > 0
    ]
    .groupby("low_reference_flag")["csi"]
    .describe()
)

print("=" * 70)

LOW CLEAR-SKY REFERENCE FLAG

Flagged observations:
4581

Percentage of all observations:
5.223727422003284

Percentage of positive-reference observations:
9.617887885786269

CSI statistics by reference category:
                      count      mean       std       min       25%       50%  \
low_reference_flag                                                              
False               43049.0  0.672394  0.311975  0.061697  0.386520  0.753699   
True                 4581.0  0.770544  0.336406  0.103028  0.563392  1.000000   

                        75%      max  
low_reference_flag                    
False               0.99395  1.00000  
True                1.00000  1.00082  


In [21]:
low_ref_df = df[
    df["low_reference_flag"]
].copy()

low_ref_df["hour"] = low_ref_df["start_time"].dt.hour

print("=" * 70)
print("LOW-REFERENCE OBSERVATIONS BY HOUR")
print("=" * 70)

print("\nCounts by hour:")
print(
    low_ref_df["hour"]
    .value_counts()
    .sort_index()
)

print("\nPercentage of positive-reference observations by hour:")

hour_total = (
    df[df["clear_sky_ghi"] > 0]
    .assign(hour=lambda x: x["start_time"].dt.hour)
    .groupby("hour")
    .size()
)

hour_low = (
    low_ref_df
    .groupby("hour")
    .size()
)

hour_summary = pd.DataFrame({
    "positive_reference_count": hour_total,
    "low_reference_count": hour_low
}).fillna(0)

hour_summary["low_reference_percentage"] = (
    100 *
    hour_summary["low_reference_count"] /
    hour_summary["positive_reference_count"]
)

print(hour_summary)

print("=" * 70)

LOW-REFERENCE OBSERVATIONS BY HOUR

Counts by hour:
hour
3     778
4     507
5     436
6     521
15    450
16    434
17    433
18    530
19    492
Name: count, dtype: int64

Percentage of positive-reference observations by hour:
      positive_reference_count  low_reference_count  low_reference_percentage
hour                                                                         
3                          778                778.0                100.000000
4                         1607                507.0                 31.549471
5                         2354                436.0                 18.521665
6                         3178                521.0                 16.393958
7                         3654                  0.0                  0.000000
8                         3654                  0.0                  0.000000
9                         3654                  0.0                  0.000000
10                        3654                  0.0                  

## Low Clear-Sky Reference Assessment

A diagnostic flag was created for observations with a positive but small clear-sky GHI reference:

$$
0 < GHI_{\mathrm{clear}} \leq 20\ \mathrm{Wh/m^2}
$$

A total of 4,581 observations satisfy this condition, corresponding to 5.22% of the complete dataset and 9.62% of positive-reference observations.

The low-reference observations are strongly concentrated at the temporal boundaries of the positive-reference period. They occur at hours 03–06 and 15–19 in the downloaded dataset, with the proportion reaching 100% at hours 03 and 19.

The CSI distribution of the low-reference group differs from that of observations with larger clear-sky references. The low-reference group has a mean CSI of 0.7705 and median of 1.0, while the non-low-reference group has a mean CSI of 0.6724 and median of 0.7537.

These observations are therefore not automatically classified as invalid or excluded. The low-reference condition is retained as a diagnostic flag for sensitivity analysis.

The threshold of 20 Wh/m² is used only for this diagnostic flag and is not used as a drought threshold or as a data-exclusion criterion.

In [22]:
from pathlib import Path

output_dir = Path("../data/interim")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "solar_historical_2015_2025_validated.csv"

df.to_csv(output_path, index=False)

print("=" * 70)
print("VALIDATED HISTORICAL DATASET SAVED")
print("=" * 70)

print("\nPath:")
print(output_path)

print("\nExists:")
print(output_path.exists())

print("\nShape:")
print(df.shape)

print("\nFile size (MB):")
print(round(output_path.stat().st_size / (1024**2), 3))

print("=" * 70)

VALIDATED HISTORICAL DATASET SAVED

Path:
..\data\interim\solar_historical_2015_2025_validated.csv

Exists:
True

Shape:
(87696, 22)

File size (MB):
17.839


# Conclusion

The complete historical solar dataset was successfully prepared for subsequent anomaly and drought analysis.

The dataset contains 87,696 hourly observations spanning 2015-01-01 00:00 through 2025-01-02 00:00. Temporal quality assessment identified no duplicate timestamps, unexpected observation durations, or missing hourly intervals.

No missing numeric radiation values or negative radiation observations were identified. The measured GHI was internally consistent with the measured beam and diffuse components, with no observations exceeding the predefined 0.01 Wh/m² component-balance tolerance.

The source reliability variable was retained as a quality-control variable rather than used for automatic observation removal. Reduced reliability was concentrated primarily within positive-reference observations, so excluding such observations a priori could alter the lower tail of the solar-availability distribution.

Solar observations were classified using the clear-sky GHI reference:

- clear-sky GHI = 0: nighttime
- clear-sky GHI > 0: positive-reference period

This produced 40,066 nighttime observations and 47,630 positive-reference observations.

For positive-reference observations, the clear-sky index was calculated as:

$$
CSI_t =
\frac{GHI_t}{GHI_{\mathrm{clear},t}}
$$

Nighttime observations were assigned an undefined CSI because the ratio is mathematically undefined when both observed and clear-sky GHI are zero.

The resulting historical CSI population contains 47,630 observations with a mean of 0.681834 and standard deviation of 0.315732. The minimum CSI was 0.061697 and the 10th percentile was 0.179170.

Only one observation produced a CSI greater than 1. This observation had a very small absolute difference between observed and clear-sky GHI and had full source reliability; it was therefore retained without clipping or modification.

CSI reconstruction reproduced the original GHI values to floating-point precision, confirming the correctness of the normalization implementation.

Observations with small positive clear-sky GHI references were retained and marked using a diagnostic low-reference flag. This flag is not used as a data-exclusion criterion or drought definition. Its influence can be evaluated later through sensitivity analysis.

The validated historical dataset has been saved to:

`data/interim/solar_historical_2015_2025_validated.csv`

This dataset provides the input for the subsequent statistical anomaly-detection analysis.